In [2]:
import polars as pl
import pandas as pd

print("Polars version:", pl.__version__)

Polars version: 1.38.1


In [3]:
# Creating a DataFrame in Polars - very similar to Pandas
df = pl.DataFrame({
    "name":    ["Alice", "Bob", "Carol", "Dave"],
    "revenue": [1200, 800, 1500, 600],
    "region":  ["North", "South", "North", "South"]
})

print(df)
print("\nShape:", df.shape)
print("\nSchema:", df.schema)  # Polars calls it schema, not dtypes

shape: (4, 3)
┌───────┬─────────┬────────┐
│ name  ┆ revenue ┆ region │
│ ---   ┆ ---     ┆ ---    │
│ str   ┆ i64     ┆ str    │
╞═══════╪═════════╪════════╡
│ Alice ┆ 1200    ┆ North  │
│ Bob   ┆ 800     ┆ South  │
│ Carol ┆ 1500    ┆ North  │
│ Dave  ┆ 600     ┆ South  │
└───────┴─────────┴────────┘

Shape: (4, 3)

Schema: Schema([('name', String), ('revenue', Int64), ('region', String)])


In [4]:
# EAGER — executes immediately, like Pandas
# Use for: small data, exploration, quick checks
eager_result = df.filter(pl.col("revenue") > 1000)
print("Eager result:")
print(eager_result)

# LAZY — builds a query plan, executes only when you call .collect()
# Use for: large data, production pipelines
lazy_result = (
    df.lazy()
    .filter(pl.col("revenue") > 1000)
    .select(["name", "revenue"])
)

print("\nLazy plan (not executed yet):")
print(lazy_result)   # just shows the plan

print("\nLazy result after .collect():")
print(lazy_result.collect())  # NOW it executes

Eager result:
shape: (2, 3)
┌───────┬─────────┬────────┐
│ name  ┆ revenue ┆ region │
│ ---   ┆ ---     ┆ ---    │
│ str   ┆ i64     ┆ str    │
╞═══════╪═════════╪════════╡
│ Alice ┆ 1200    ┆ North  │
│ Carol ┆ 1500    ┆ North  │
└───────┴─────────┴────────┘

Lazy plan (not executed yet):
naive plan: (run LazyFrame.explain(optimized=True) to see the optimized plan)

SELECT [col("name"), col("revenue")]
  FILTER [(col("revenue")) > (1000)]
  FROM
    DF ["name", "revenue", "region"]; PROJECT */3 COLUMNS

Lazy result after .collect():
shape: (2, 2)
┌───────┬─────────┐
│ name  ┆ revenue │
│ ---   ┆ ---     │
│ str   ┆ i64     │
╞═══════╪═════════╡
│ Alice ┆ 1200    │
│ Carol ┆ 1500    │
└───────┴─────────┘


In [5]:
# In Pandas you reference columns like this:
# df["revenue"] or df[df["revenue"] > 1000]

# In Polars you use pl.col() — expressions
# This is the single biggest syntax difference

# Filtering
print("Filter:")
print(df.filter(pl.col("revenue") > 1000))

# Selecting columns
print("\nSelect:")
print(df.select(["name", "revenue"]))

# Adding a new column — with_columns instead of df["new"] =
print("\nAdd column:")
print(df.with_columns(
    (pl.col("revenue") * 1.10).alias("revenue_with_bonus")
))

# Multiple operations at once
print("\nMultiple new columns at once:")
print(df.with_columns([
    (pl.col("revenue") * 1.10).alias("revenue_with_bonus"),
    (pl.col("revenue") / 1000).alias("revenue_thousands"),
    pl.lit("Active").alias("status")   # pl.lit() for literal values
]))

Filter:
shape: (2, 3)
┌───────┬─────────┬────────┐
│ name  ┆ revenue ┆ region │
│ ---   ┆ ---     ┆ ---    │
│ str   ┆ i64     ┆ str    │
╞═══════╪═════════╪════════╡
│ Alice ┆ 1200    ┆ North  │
│ Carol ┆ 1500    ┆ North  │
└───────┴─────────┴────────┘

Select:
shape: (4, 2)
┌───────┬─────────┐
│ name  ┆ revenue │
│ ---   ┆ ---     │
│ str   ┆ i64     │
╞═══════╪═════════╡
│ Alice ┆ 1200    │
│ Bob   ┆ 800     │
│ Carol ┆ 1500    │
│ Dave  ┆ 600     │
└───────┴─────────┘

Add column:
shape: (4, 4)
┌───────┬─────────┬────────┬────────────────────┐
│ name  ┆ revenue ┆ region ┆ revenue_with_bonus │
│ ---   ┆ ---     ┆ ---    ┆ ---                │
│ str   ┆ i64     ┆ str    ┆ f64                │
╞═══════╪═════════╪════════╪════════════════════╡
│ Alice ┆ 1200    ┆ North  ┆ 1320.0             │
│ Bob   ┆ 800     ┆ South  ┆ 880.0              │
│ Carol ┆ 1500    ┆ North  ┆ 1650.0             │
│ Dave  ┆ 600     ┆ South  ┆ 660.0              │
└───────┴─────────┴────────┴──────────────────

In [6]:
df2 = pl.DataFrame({
    "name":     ["Alice", "Bob", "Carol", "Dave", "Eve"],
    "region":   ["North", "South", "North", "South", "North"],
    "revenue":  [1200, 800, 1500, 600, 900],
    "category": ["A", "B", "A", "B", "B"]
})

# Basic groupby
print("Total revenue per region:")
print(df2.group_by("region").agg(pl.col("revenue").sum()))

# Multiple aggregations at once
print("\nMultiple aggregations:")
print(df2.group_by("region").agg([
    pl.col("revenue").sum().alias("total_revenue"),
    pl.col("revenue").mean().alias("avg_revenue"),
    pl.col("revenue").count().alias("count")
]))

# Group by multiple columns
print("\nGroup by region and category:")
print(df2.group_by(["region", "category"]).agg(
    pl.col("revenue").sum().alias("total_revenue")
))

Total revenue per region:
shape: (2, 2)
┌────────┬─────────┐
│ region ┆ revenue │
│ ---    ┆ ---     │
│ str    ┆ i64     │
╞════════╪═════════╡
│ South  ┆ 1400    │
│ North  ┆ 3600    │
└────────┴─────────┘

Multiple aggregations:
shape: (2, 4)
┌────────┬───────────────┬─────────────┬───────┐
│ region ┆ total_revenue ┆ avg_revenue ┆ count │
│ ---    ┆ ---           ┆ ---         ┆ ---   │
│ str    ┆ i64           ┆ f64         ┆ u32   │
╞════════╪═══════════════╪═════════════╪═══════╡
│ North  ┆ 3600          ┆ 1200.0      ┆ 3     │
│ South  ┆ 1400          ┆ 700.0       ┆ 2     │
└────────┴───────────────┴─────────────┴───────┘

Group by region and category:
shape: (3, 3)
┌────────┬──────────┬───────────────┐
│ region ┆ category ┆ total_revenue │
│ ---    ┆ ---      ┆ ---           │
│ str    ┆ str      ┆ i64           │
╞════════╪══════════╪═══════════════╡
│ South  ┆ B        ┆ 1400          │
│ North  ┆ A        ┆ 2700          │
│ North  ┆ B        ┆ 900           │
└────────┴───

In [7]:
customers = pl.DataFrame({
    "customer_id": [1, 2, 3, 4],
    "name": ["Alice", "Bob", "Carol", "Dave"]
})

orders = pl.DataFrame({
    "order_id":    [101, 102, 103, 104, 105],
    "customer_id": [1, 2, 2, 3, 5],
    "amount":      [250, 150, 300, 175, 200]
})

# Left join — cleaner syntax than Pandas
print("Left join:")
print(customers.join(orders, on="customer_id", how="left"))

# Inner join
print("\nInner join:")
print(customers.join(orders, on="customer_id", how="inner"))

Left join:
shape: (5, 4)
┌─────────────┬───────┬──────────┬────────┐
│ customer_id ┆ name  ┆ order_id ┆ amount │
│ ---         ┆ ---   ┆ ---      ┆ ---    │
│ i64         ┆ str   ┆ i64      ┆ i64    │
╞═════════════╪═══════╪══════════╪════════╡
│ 1           ┆ Alice ┆ 101      ┆ 250    │
│ 2           ┆ Bob   ┆ 102      ┆ 150    │
│ 2           ┆ Bob   ┆ 103      ┆ 300    │
│ 3           ┆ Carol ┆ 104      ┆ 175    │
│ 4           ┆ Dave  ┆ null     ┆ null   │
└─────────────┴───────┴──────────┴────────┘

Inner join:
shape: (4, 4)
┌─────────────┬───────┬──────────┬────────┐
│ customer_id ┆ name  ┆ order_id ┆ amount │
│ ---         ┆ ---   ┆ ---      ┆ ---    │
│ i64         ┆ str   ┆ i64      ┆ i64    │
╞═════════════╪═══════╪══════════╪════════╡
│ 1           ┆ Alice ┆ 101      ┆ 250    │
│ 2           ┆ Bob   ┆ 102      ┆ 150    │
│ 2           ┆ Bob   ┆ 103      ┆ 300    │
│ 3           ┆ Carol ┆ 104      ┆ 175    │
└─────────────┴───────┴──────────┴────────┘
